# Artificial Intelligence — Exercise 1
## International Football Results Analysis (1872–2024)

**Dataset:** [Kaggle — International Football Results](https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017)

This notebook explores international football match data through basic data exploration, goals analysis, match result classification, and visualizations.

## Step 1: Load the CSV

We begin by importing `pandas` and loading the dataset into a DataFrame. The `head()` call lets us preview the first few rows to understand the structure.

In [ ]:
import pandas as pd

df = pd.read_csv("results.csv")
df.head()

---
## Part 1: Basic Exploration

Before diving into analysis, we need to understand the shape and content of our dataset — how many rows, the date range, the number of countries, and which team is most frequently the home side.

### Q1: How many matches are in the dataset?

`df.shape` returns a tuple `(rows, columns)`. The first value gives us the total number of matches (each row = one match).

In [ ]:
total_matches = df.shape[0]
print(f"Total number of matches: {total_matches}")

### Q2: What is the earliest and latest year in the data?

The `date` column is first converted to a proper datetime type so we can extract the year. `.min()` and `.max()` then give us the earliest and latest dates.

In [ ]:
df["date"] = pd.to_datetime(df["date"])

earliest = df["date"].min()
latest   = df["date"].max()

print(f"Earliest match: {earliest.date()}")
print(f"Latest match:   {latest.date()}")

### Q3: How many unique countries are there?

Countries appear in both the `home_team` and `away_team` columns. We combine both columns into a single set and count the unique entries.

In [ ]:
all_teams = pd.concat([df["home_team"], df["away_team"]])
unique_countries = all_teams.nunique()

print(f"Number of unique countries/teams: {unique_countries}")

### Q4: Which team appears most frequently as the home team?

`value_counts()` counts occurrences of each value. `.head(10)` shows the top 10 home teams by frequency.

In [ ]:
top_home_teams = df["home_team"].value_counts().head(10)
print("Top 10 most frequent home teams:")
print(top_home_teams)

---
## Part 2: Goals Analysis

We create a new column `total_goals` by summing the home and away scores for each match, then use it to explore scoring patterns.

In [ ]:
df["total_goals"] = df["home_score"] + df["away_score"]
df[["home_team", "away_team", "home_score", "away_score", "total_goals"]].head()

### Q5: What is the average number of goals per match?

`.mean()` computes the arithmetic mean of `total_goals` across all matches.

In [ ]:
avg_goals = df["total_goals"].mean()
print(f"Average goals per match: {avg_goals:.2f}")

### Q6: What is the highest scoring match?

`.idxmax()` finds the index of the row with the maximum `total_goals`. We then display that row for full context.

In [ ]:
highest_idx = df["total_goals"].idxmax()
highest_match = df.loc[highest_idx]
print("Highest scoring match:")
print(highest_match[["date", "home_team", "away_team", "home_score", "away_score", "total_goals"]])

### Q7: Are more goals scored at home or away?

We sum all home scores and all away scores separately and compare them to determine where most goals come from.

In [ ]:
total_home_goals = df["home_score"].sum()
total_away_goals = df["away_score"].sum()

print(f"Total home goals: {total_home_goals}")
print(f"Total away goals: {total_away_goals}")

if total_home_goals > total_away_goals:
    print("→ More goals are scored at HOME.")
else:
    print("→ More goals are scored AWAY.")

### Q8: What is the most common total goals value?

`.mode()[0]` returns the most frequently occurring value in `total_goals`, i.e., the most common final scoreline total.

In [ ]:
most_common_goals = df["total_goals"].mode()[0]
frequency = (df["total_goals"] == most_common_goals).sum()

print(f"Most common total goals per match: {most_common_goals}")
print(f"This occurred in {frequency} matches.")

---
## Part 3: Match Results

We classify every match into one of three outcomes — **Home Win**, **Away Win**, or **Draw** — using a custom function applied row by row.

In [ ]:
def match_result(row):
    if row["home_score"] > row["away_score"]:
        return "Home Win"
    elif row["home_score"] < row["away_score"]:
        return "Away Win"
    else:
        return "Draw"

df["result"] = df.apply(match_result, axis=1)
df[["home_team", "away_team", "home_score", "away_score", "result"]].head()

### Q9: What percentage of matches are home wins?

We count how many matches resulted in each outcome and convert to percentages.

In [ ]:
result_counts = df["result"].value_counts()
result_pct    = df["result"].value_counts(normalize=True) * 100

print("Match outcome breakdown:")
for outcome in result_counts.index:
    print(f"  {outcome}: {result_counts[outcome]} matches ({result_pct[outcome]:.1f}%)")

### Q10: Does home advantage exist?

Home advantage exists if the percentage of **Home Wins** is notably higher than **Away Wins**.

In [ ]:
home_win_pct = result_pct.get("Home Win", 0)
away_win_pct = result_pct.get("Away Win", 0)

print(f"Home Win %: {home_win_pct:.1f}%")
print(f"Away Win %: {away_win_pct:.1f}%")

if home_win_pct > away_win_pct:
    print(f"\n✅ Home advantage EXISTS — home teams win {home_win_pct - away_win_pct:.1f}% more often.")
else:
    print("\n❌ No clear home advantage in this dataset.")

### Q11: Which country has the most wins historically?

A team wins when it is the home team and the result is 'Home Win', **or** it is the away team and the result is 'Away Win'. We union both sets of wins and count.

In [ ]:
home_wins = df[df["result"] == "Home Win"]["home_team"]
away_wins = df[df["result"] == "Away Win"]["away_team"]

all_wins = pd.concat([home_wins, away_wins])
top_winners = all_wins.value_counts().head(10)

print("Top 10 countries by total wins:")
print(top_winners)

---
## Part 4: Visualization

We use `matplotlib` to create three charts that summarise our key findings visually.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

### Chart 1: Histogram of Goals Per Match

A histogram shows the frequency distribution of `total_goals`. We expect most matches to have a low-to-moderate number of goals, with very high-scoring matches being rare (right-skewed distribution).

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["total_goals"], bins=20, color="steelblue", edgecolor="white")
plt.title("Distribution of Total Goals Per Match", fontsize=14, fontweight="bold")
plt.xlabel("Total Goals")
plt.ylabel("Number of Matches")
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.savefig("goals_histogram.png", dpi=150)
plt.show()

### Chart 2: Bar Chart of Match Outcomes

This chart compares the count of Home Wins, Away Wins, and Draws — making it easy to see which outcome is most common and whether home advantage is visible.

In [ ]:
outcome_counts = df["result"].value_counts()
colors = ["#2ecc71", "#e74c3c", "#95a5a6"]

plt.figure(figsize=(7, 5))
bars = plt.bar(outcome_counts.index, outcome_counts.values, color=colors, edgecolor="white")
plt.title("Match Outcomes", fontsize=14, fontweight="bold")
plt.xlabel("Outcome")
plt.ylabel("Number of Matches")

# Annotate each bar with its count
for bar in bars:
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 100,
             f"{bar.get_height():,}",
             ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("match_outcomes.png", dpi=150)
plt.show()

### Chart 3: Top 10 Teams by Total Wins

A horizontal bar chart makes it easy to read team names. We use the `all_wins` series computed earlier.

In [ ]:
top10 = all_wins.value_counts().head(10).sort_values()

plt.figure(figsize=(9, 6))
plt.barh(top10.index, top10.values, color="darkorange", edgecolor="white")
plt.title("Top 10 Countries by Total Wins", fontsize=14, fontweight="bold")
plt.xlabel("Total Wins")
plt.tight_layout()
plt.savefig("top10_wins.png", dpi=150)
plt.show()

---
## Summary

| Question | Finding |
|---|---|
| Total matches | See output above |
| Date range | 1872 – 2024 |
| Unique countries | See output above |
| Most frequent home team | See output above |
| Average goals/match | See output above |
| Highest scoring match | See output above |
| Goals: home vs away | Home teams score more on average |
| Most common scoreline total | See output above |
| Home win % | See output above |
| Home advantage | Yes — home teams win more often |
| Most wins historically | See output above |